# 03_limpieza_datos

Proyecto ARIMA / ARIMAX
Modelación epidemiológica con variables meteorológicas.

# Obtención de los datos epidemiológicos 

In [2]:
# Inicio del análisis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  


In [4]:
# Importar datos epidemiológicos 
ubicacion_epi_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Secretaria de salud\BD_DENGUE_2021-2025_CAUCASIA.xlsx"
ubicacion_epi_marco = r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\5_arima\6_datos_arima\3_datos_version_6_abril_2026\epidemiologicos.xlsx"
df_epi_caucasia=pd.read_excel(ubicacion_epi_janis)

# Importar datos meteorológicos 
ubicacion_meteo_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Datos meteorológicos\Datos_NS_2021-2025_SOI-SST.xlsx"
ubicacion_meteo_marco = r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\5_arima\6_datos_arima\3_datos_version_6_abril_2026\meteorologicos.xlsx"
df_meteo_caucasia=pd.read_excel(ubicacion_meteo_janis) 



In [5]:
df_epi_caucasia.head(2)

,ini_sin_,semana,año,area_,localidad_,cen_pobla_,vereda_,bar_ver_,dir_res_,nmun_proce,nmun_resi
0,2021-01-21,3,2021,2,NO APLICA,CAUCASIA,NO APLICA,CAUCASIA,CR 39 E 48 C SUR 36,CAUCASIA,ENVIGADO
1,2021-02-08,6,2021,2,NO APLICA,CAUCASIA,NO APLICA,CAUCASIA,CL 29 A 42-99,CAUCASIA,BOGOTA


In [6]:
df_meteo_caucasia.head(2)

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,QV2M,RH2M,PRECTOTCORR,WS2M,WS2M_MAX,WS2M_MIN,ALLSKY_SFC_UV_INDEX,SOI,SST_ANOMALY
0,2021,3,28.33,35.09,23.84,18.15,78.13,5.83,0.13,0.32,0.07,2.48,23.42,-1.038442
1,2021,4,28.71,35.05,23.69,16.29,69.26,0.77,0.14,0.30,0.07,2.47,25.40,-1.071886


In [7]:
df_meteo_caucasia.columns

Index(['YEAR', 'DOY', 'T2M', 'T2M_MAX', 'T2M_MIN', 'QV2M', 'RH2M',
       'PRECTOTCORR', 'WS2M', 'WS2M_MAX', 'WS2M_MIN', 'ALLSKY_SFC_UV_INDEX',
       'SOI', 'SST_ANOMALY'],
      dtype='object')

In [8]:
df_epi_caucasia.columns 

Index(['ini_sin_', 'semana', 'año', 'area_', 'localidad_', 'cen_pobla_',
       'vereda_', 'bar_ver_', 'dir_res_', 'nmun_proce', 'nmun_resi'],
      dtype='object')

# Remuestreo de los datos Epidemiológicos a frecuencia semanal 

In [9]:
# remuestreo epidemiológico 2024 y 2025. 2021 al 2024 con 52 semanas, 2025 con 53 semanas
# ---------------------------
# 1. AGRUPAR CASOS POR SEMANA REAL
# ---------------------------
df_epi = df_epi_caucasia.copy()

df_semanal = df_epi.groupby(['año', 'semana']).size().reset_index(name='num_casos')


# ---------------------------
# 2. ESTRUCTURA 2021–2024 (52 semanas)
# ---------------------------
estructura_1 = pd.MultiIndex.from_product(
    [range(2021, 2025), range(1, 53)],
    names=['año', 'semana']
).to_frame(index=False)

df_1_epi = pd.merge(estructura_1, df_semanal, on=['año', 'semana'], how='left')
df_1_epi['num_casos'] = df_1_epi['num_casos'].fillna(0).astype(int)


# ---------------------------
# 3. ESTRUCTURA 2025 (53 semanas)
# ---------------------------
estructura_2 = pd.DataFrame({
    'año': [2025]*53,
    'semana': range(1, 54)
})

df_2_epi = pd.merge(estructura_2, df_semanal, on=['año', 'semana'], how='left')
df_2_epi['num_casos'] = df_2_epi['num_casos'].fillna(0).astype(int)


# ---------------------------
# 4. FECHAS CORRECTAS (CLAVE)
# ---------------------------
inicios = {
    2021: '2021-01-03',
    2022: '2022-01-02',
    2023: '2023-01-01',
    2024: '2023-12-31',  # ← corrige el error de 25 vs 22 dic
    2025: '2024-12-29'
}

def asignar_fecha(row):
    inicio = pd.Timestamp(inicios[row['año']])
    return inicio + pd.to_timedelta((row['semana'] - 1) * 7, unit='D')


# Aplicar
df_1_epi['fecha'] = df_1_epi.apply(asignar_fecha, axis=1)
df_2_epi['fecha'] = df_2_epi.apply(asignar_fecha, axis=1)


# ---------------------------
# 5. UNIR TODO
# ---------------------------
df_final_epi = pd.concat([df_1_epi, df_2_epi])

df_final_epi = df_final_epi.sort_values('fecha').set_index('fecha')
df_final_epi = df_final_epi.rename(columns={'semana': 'semana_epi'})

In [10]:
df_1_epi

,año,semana,num_casos,fecha
0,2021,1,0,2021-01-03
1,2021,2,0,2021-01-10
2,2021,3,1,2021-01-17
3,2021,4,0,2021-01-24
4,2021,5,0,2021-01-31
...,...,...,...,...
203,2024,48,68,2024-11-24
204,2024,49,92,2024-12-01
205,2024,50,85,2024-12-08
206,2024,51,74,2024-12-15


In [11]:
df_2_epi

,año,semana,num_casos,fecha
0,2025,1,92,2024-12-29
1,2025,2,90,2025-01-05
2,2025,3,82,2025-01-12
3,2025,4,99,2025-01-19
4,2025,5,103,2025-01-26
5,2025,6,106,2025-02-02
6,2025,7,71,2025-02-09
7,2025,8,78,2025-02-16
8,2025,9,52,2025-02-23
9,2025,10,83,2025-03-02


### Meteorológicos

In [12]:
# Renombrar columnas del dataframe nasa
df_meteo_caucasia.rename(columns={
    'YEAR': 'año',
    'DOY': 'dia',
    'T2M': 'temp',
    'T2M_MAX': 'temp_max',
    'T2M_MIN': 'temp_min',
    'QV2M': 'hum_esp',
    'RH2M': 'hum_rel',
    'PRECTOTCORR': 'prec',
    'WS2M': 'vel_vi',
    'WS2M_MAX': 'vel_vi_max',
    'WS2M_MIN': 'vel_vi_min',
    'ALLSKY_SFC_UV_INDEX': 'uv',
    'SOI': 'soi',
    'SST_ANOMALY':'sst'
}, inplace=True)
df_meteo_caucasia.head(3)

,año,dia,temp,temp_max,temp_min,hum_esp,hum_rel,prec,vel_vi,vel_vi_max,vel_vi_min,uv,soi,sst
0,2021,3,28.33,35.09,23.84,18.15,78.13,5.83,0.13,0.32,0.07,2.48,23.42,-1.038442
1,2021,4,28.71,35.05,23.69,16.29,69.26,0.77,0.14,0.30,0.07,2.47,25.40,-1.071886
2,2021,5,27.85,34.96,22.71,15.36,69.34,0.80,0.16,0.34,0.03,2.30,20.07,-1.103086


In [13]:
df_meteo_caucasia.columns

Index(['año', 'dia', 'temp', 'temp_max', 'temp_min', 'hum_esp', 'hum_rel',
       'prec', 'vel_vi', 'vel_vi_max', 'vel_vi_min', 'uv', 'soi', 'sst'],
      dtype='object')

In [14]:
df_meteo = df_meteo_caucasia.copy()

# Crear fecha real
df_meteo['fecha'] = pd.to_datetime(df_meteo['año'].astype(str) + '-' + df_meteo['dia'].astype(str), format='%Y-%j')

In [15]:
df_meteo['dias_lluvia'] = (df_meteo['prec'] >= 1).astype(int)

In [16]:
inicios = {
    2021: pd.Timestamp('2021-01-03'),
    2022: pd.Timestamp('2022-01-02'),
    2023: pd.Timestamp('2023-01-01'),
    2024: pd.Timestamp('2023-12-31'),
    2025: pd.Timestamp('2024-12-29')
}

def asignar_semana(row):
    año = row['fecha'].year
    
    # Ajuste especial para fechas que pertenecen a semana epi del siguiente año
    if row['fecha'] >= inicios[2025]:
        año = 2025
    elif row['fecha'] >= inicios[2024]:
        año = 2024
    elif row['fecha'] >= inicios[2023]:
        año = 2023
    elif row['fecha'] >= inicios[2022]:
        año = 2022
    else:
        año = 2021

    semana = ((row['fecha'] - inicios[año]).days // 7) + 1
    return pd.Series([año, semana])

df_meteo[['año_epi', 'semana']] = df_meteo.apply(asignar_semana, axis=1)

In [17]:
df_semanal = df_meteo.groupby(['año_epi', 'semana']).agg({
    'temp': 'mean',
    'temp_max': 'mean',
    'temp_min': 'mean',
    'hum_esp': 'mean',
    'hum_rel': 'mean',
    'prec': 'mean',
    'dias_lluvia': 'sum',
    'vel_vi': 'mean',
    'vel_vi_max': 'mean',
    'vel_vi_min': 'mean',
    'uv': 'mean',
    'prec': 'sum',
    'soi': 'mean',
    'sst': 'mean'
}).reset_index()

df_semanal = df_semanal.rename(columns={'año_epi': 'año'})

In [18]:
estructura_1 = pd.MultiIndex.from_product(
    [range(2021, 2025), range(1, 53)],
    names=['año', 'semana']
).to_frame(index=False)

df_1_meteo = pd.merge(estructura_1, df_semanal, on=['año', 'semana'], how='left')

In [19]:
estructura_2 = pd.DataFrame({
    'año': [2025]*53,
    'semana': range(1, 54)
})

df_2_meteo = pd.merge(estructura_2, df_semanal, on=['año', 'semana'], how='left')

In [20]:
def asignar_fecha(row):
    inicio = inicios[row['año']]
    return inicio + pd.to_timedelta((row['semana'] - 1) * 7, unit='D')

df_1_meteo['fecha'] = df_1_meteo.apply(asignar_fecha, axis=1)
df_2_meteo['fecha'] = df_2_meteo.apply(asignar_fecha, axis=1)

In [21]:
df_final_meteo = pd.concat([df_1_meteo, df_2_meteo])
df_final_meteo = df_final_meteo.sort_values('fecha').set_index('fecha')

In [22]:
df_1_meteo

,año,semana,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv,soi,sst,fecha
0,2021,1,28.254286,34.200000,23.832857,16.367143,70.774286,7.73,1,0.124286,0.268571,0.047143,2.222857,20.067143,-1.060214,2021-01-03
1,2021,2,28.638571,34.910000,24.195714,17.318571,73.095714,19.18,5,0.111429,0.204286,0.038571,2.254286,18.207143,-1.058352,2021-01-10
2,2021,3,29.552857,36.372857,24.090000,16.122857,65.328571,0.80,0,0.122857,0.227143,0.045714,2.420000,10.847143,-0.931407,2021-01-17
3,2021,4,29.208571,35.978571,24.200000,16.564286,68.205714,12.81,5,0.122857,0.220000,0.035714,2.477143,15.430000,-0.837360,2021-01-24
4,2021,5,29.421429,35.882857,24.684286,17.311429,69.797143,18.06,3,0.117143,0.225714,0.030000,2.290000,9.348571,-0.897892,2021-01-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,2024,48,26.278571,29.455714,24.164286,19.707143,91.838571,43.67,7,0.157143,0.352857,0.040000,1.878571,16.867143,0.090169,2024-11-24
204,2024,49,26.570000,29.908571,24.212857,19.700000,90.314286,18.82,4,0.150000,0.322857,0.054286,1.762857,18.444286,-0.083963,2024-12-01
205,2024,50,26.778571,30.527143,23.577143,19.062857,86.507143,5.85,1,0.195714,0.518571,0.021429,1.921429,6.710000,-0.382062,2024-12-08
206,2024,51,26.618571,29.824286,24.062857,19.502857,89.172857,11.85,3,0.161429,0.472857,0.021429,1.652857,14.620000,-0.718384,2024-12-15


In [23]:
df_2_meteo

,año,semana,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv,soi,sst,fecha
0,2025,1,25.158571,28.048571,23.124286,18.602857,92.682857,73.12,7,0.158571,0.400000,0.030000,1.744286,-3.594286,-0.703858,2024-12-29
1,2025,2,26.551429,30.398571,23.624286,18.697143,86.357143,13.31,2,0.178571,0.421429,0.040000,1.871429,-4.187143,-0.750312,2025-01-05
2,2025,3,26.797143,31.061429,23.178571,18.377143,83.964286,0.80,0,0.180000,0.478571,0.032857,2.080000,-7.405714,-0.698161,2025-01-12
3,2025,4,26.945714,31.498571,23.451429,18.271429,82.832857,2.09,1,0.168571,0.460000,0.008571,1.988571,11.404286,-0.978982,2025-01-19
4,2025,5,27.128571,31.527143,24.010000,18.997143,84.857143,11.13,2,0.120000,0.272857,0.017143,1.947143,14.830000,-0.951556,2025-01-26
5,2025,6,27.062857,31.348571,23.768571,18.447143,82.987143,35.83,5,0.132857,0.320000,0.022857,1.775714,25.092857,-0.671806,2025-02-02
6,2025,7,26.912857,31.570000,23.697143,18.330000,83.498571,33.52,7,0.124286,0.240000,0.024286,1.754286,4.047143,-0.433545,2025-02-09
7,2025,8,27.505714,32.497143,24.010000,18.398571,81.260000,16.48,6,0.140000,0.290000,0.042857,1.890000,-7.480000,-0.278124,2025-02-16
8,2025,9,27.672857,32.394286,24.411429,18.815714,81.998571,25.08,5,0.134286,0.228571,0.060000,1.805714,4.460000,-0.186427,2025-02-23
9,2025,10,27.247143,32.092857,24.110000,18.282857,81.888571,23.79,6,0.137143,0.238571,0.060000,1.942857,4.984286,0.046518,2025-03-02


# Unir las bases de datos epidemiológicas con la base de datos meteorológica de acuerdo al criterio año semana epidemiológica 


In [25]:
# Unir datos
datos_semanal_meteo_epi = (
    pd.merge(
        df_final_epi.reset_index(),
        df_final_meteo.reset_index().drop(columns=['año', 'semana']),
        on='fecha',
        how='left'
    )
    .rename(columns={'num_casos': 'casos_dengue'})
)

# Quitar hora de la fecha
datos_semanal_meteo_epi['fecha'] = pd.to_datetime(datos_semanal_meteo_epi['fecha']).dt.date

# Reordenar columnas
columnas_orden = [
    'fecha',
    'año',
    'semana_epi',
    'temp', 'temp_max', 'temp_min',
    'hum_esp', 'hum_rel',
    'prec', 'dias_lluvia',
    'vel_vi', 'vel_vi_max', 'vel_vi_min',
    'uv', 'soi', 'sst', 'casos_dengue'
]

datos_semanal_meteo_epi = datos_semanal_meteo_epi[columnas_orden]

# Guardar
guardar_ubicacion_marco = r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\5_arima\6_datos_arima\4_base_datos_consolidada\datos_semanal_meteo_epi.xlsx"
guardar_ubicacion_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Datos consolidados\datos_semanal_meteo_epi.xlsx"
datos_semanal_meteo_epi.to_excel(guardar_ubicacion_janis, index=False)

datos_semanal_meteo_epi.head()

,fecha,año,semana_epi,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv,soi,sst,casos_dengue
0,2021-01-03,2021,1,28.254286,34.200000,23.832857,16.367143,70.774286,7.73,1,0.124286,0.268571,0.047143,2.222857,20.067143,-1.060214,0
1,2021-01-10,2021,2,28.638571,34.910000,24.195714,17.318571,73.095714,19.18,5,0.111429,0.204286,0.038571,2.254286,18.207143,-1.058352,0
2,2021-01-17,2021,3,29.552857,36.372857,24.090000,16.122857,65.328571,0.80,0,0.122857,0.227143,0.045714,2.420000,10.847143,-0.931407,1
3,2021-01-24,2021,4,29.208571,35.978571,24.200000,16.564286,68.205714,12.81,5,0.122857,0.220000,0.035714,2.477143,15.430000,-0.837360,0
4,2021-01-31,2021,5,29.421429,35.882857,24.684286,17.311429,69.797143,18.06,3,0.117143,0.225714,0.030000,2.290000,9.348571,-0.897892,0


Hasta aquí hemos logrado la estructuración de la base de datos epidemiológica y meteorológica.

# Adicionar al dataset los atributos meteorológicos rezagados con un rezago hasta de 16 semanas. 

In [ ]:
# Lista de variables meteorológicas
variables_meteo = ['temp', 'temp_max', 'temp_min',
       'hum_esp', 'hum_rel', 'prec', 'dias_lluvia', 'vel_vi', 'vel_vi_max',
       'vel_vi_min', 'uv', 'soi', 'sst'
]

# Número máximo de rezagos
max_lag = 16

# Iniciar el DataFrame de rezagos a partir del original
df_semanal_meteo_epi_rezagos = df_semanal_meteo_epi.copy()

# Generar rezagos
for var in variables_meteo:
    for lag in range(1, max_lag + 1):
        df_semanal_meteo_epi_rezagos[f"{var}_lag_{lag}"] = df_semanal_meteo_epi_rezagos[var].shift(lag)

# Ver resultado
df_semanal_meteo_epi_rezagos.head(15)

In [ ]:
df_semanal_meteo_epi_rezagos.columns    
df_semanal_meteo_epi_rezagos.tail(5)

In [ ]:
df_semanal_meteo_epi_rezagos.to_excel(r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\5_arima\6_datos_arima\4_base_datos_consolidada\datos_semanal_meteo_epi_rezagos.xlsx", index=False)


Tarea de resumir esta limpieza, **remuestreo** y fusion de datos, en una sola diapositiva. 